# Simulação dos testes de G1 e de H5

No começo do projeto, eu testava G1 com um binomial. A simulação abaixo mostra por que troquei pelo beta-binomial e por
que H5 passou a ser decidida só pela correção de Holm. Ela não lê nenhum arquivo: sorteia, com a semente 2026, o número
de óbitos liberados sem deriva nenhuma, de modo que toda rejeição é falsa, e leva menos de um minuto.

## Por que o teste mudou

Com $n_1$ óbitos na parte B da calibração, a regra corta no $k$-ésimo menor escore desses óbitos, com
$k = \lceil (n_1 + 1)(1 - \alpha) \rceil$. A chance de um óbito novo ser liberado não é exatamente α: depende de quais
óbitos caíram na parte B. Sob permutabilidade, e com escores contínuos, ela é $1 - U$, com
$U \sim \mathrm{Beta}(k,\ n_1 + 1 - k)$, e a média dela não passa de α. Entre $m$ óbitos do teste, o número de liberados
$X$ segue

$$X \mid U \sim \mathrm{Binomial}(m,\ 1 - U), \qquad X \sim \mathrm{BetaBinomial}(m,\ n_1 + 1 - k,\ k).$$

O teste binomial, $p = P(\mathrm{Binomial}(m, \alpha) \ge X)$, trata a FDL da regra como fixa em α e ignora a variação
que vem da calibração. O beta-binomial, $p = P(\mathrm{BetaBinomial}(m,\ n_1 + 1 - k,\ k) \ge X)$, é o que `g1_pvalue`,
em `triagem_baixo_risco/modeling/predict.py`, calcula. A diferença cresce com o número de óbitos do teste em relação ao
da parte B, porque a incerteza da regra não diminui com mais óbitos no teste.

Troquei o teste depois de ver o X2 com os dois. Com o binomial, o agregado dos 12 trimestres da regra estática refutava
H1 e H1b; com o beta-binomial, só o 1º trimestre de 2024 continua significativo. A troca veio quando eu já conhecia os
dois resultados; o que a justifica é o comportamento dos testes quando não há deriva, e é isso que a simulação mede.

Em H5 o problema era outro. A regra antiga refutava se Holm rejeitasse entre as faixas ou se alguma faixa tivesse p
abaixo de 0,05. Com um período só por faixa, como no X7c, o teste da faixa é o próprio agregado dela, e a segunda
condição anulava a correção. Holm, com quatro faixas, rejeita o $i$-ésimo menor p enquanto $p_{(i)} \le 0{,}05/(5 - i)$,
para $i$ de 1 a 4.

In [1]:
import math
from decimal import ROUND_HALF_UP, Decimal
from functools import partial

import numpy as np
import pandas as pd
import scipy
from scipy.stats import beta, betabinom, binom

from triagem_baixo_risco.modeling.predict import fdl_null

SEMENTE = 2026
ALFA = 0.05  # fração máxima de óbitos liberados pela regra
NIVEL = 0.05  # nível dos testes
R_G1 = 20_000
R_H5 = 200_000
CENARIOS_G1 = {"1. Tamanho do X7a": (2900, 1400), "2. Agregado do X2, uma calibração": (2640, 12 * 1370)}
FAIXAS_H5 = [(200, 100), (500, 250), (1100, 550), (900, 450)]  # (n1, m) de cada faixa
pd.set_option("display.max_colwidth", None)
print("numpy", np.__version__, "e scipy", scipy.__version__)


def pct(x, casas=2):
    return f"{100 * x:.{casas}f}%".replace(".", ",")


def num(x, casas=0):
    # 1234567.891 com casas=2 vira "1.234.567,89"
    return f"{x:,.{casas}f}".replace(",", " ").replace(".", ",").replace(" ", ".")

numpy 2.5.3 e scipy 1.18.1


In [2]:
def k_indice(n1):
    return math.ceil((n1 + 1) * (1 - ALFA) - 1e-12)


def sorteia_liberados(rng, n1, m, reps):
    k = k_indice(n1)
    u = beta.rvs(k, n1 + 1 - k, size=reps, random_state=rng)  # F(q1), a k-ésima estatística de ordem
    return binom.rvs(m, 1 - u, random_state=rng)


def nos_distintos(f, x):
    # o p-valor só depende de X; calcula uma vez para cada valor distinto
    valores, inverso = np.unique(x, return_inverse=True)
    return np.asarray(f(valores))[inverso]


def p_binomial(x, m):
    return nos_distintos(lambda v: binom.sf(v - 1, m, ALFA), x)


def p_betabinomial(x, n1, m):
    k = k_indice(n1)
    return nos_distintos(lambda v: betabinom.sf(v - 1, m, n1 + 1 - k, k), x)


def tamanho_exato(n1, m, pvalor):
    k = k_indice(n1)
    x = np.arange(m + 1)
    return float(np.sum(betabinom.pmf(x, m, n1 + 1 - k, k) * (pvalor(x) < NIVEL)))


def holm_rejeita(p):
    faixas = p.shape[1]
    ordem = np.argsort(p, axis=1, kind="stable")
    ordenados = np.take_along_axis(p, ordem, axis=1)
    # cumprod para no primeiro p que não passa
    segue = np.cumprod(ordenados <= NIVEL / (faixas - np.arange(faixas)), axis=1).astype(bool)
    rejeita = np.zeros_like(segue)
    np.put_along_axis(rejeita, ordem, segue, axis=1)
    return rejeita


for n1, _ in [*CENARIOS_G1.values(), *FAIXAS_H5]:
    assert fdl_null(n1, ALFA) == (n1 + 1 - k_indice(n1), k_indice(n1))

O `assert` do fim confere que os parâmetros da beta usados aqui são os mesmos de `fdl_null`, a função do pacote que
monta o nulo de G1.

## Cenários

Os cenários 1 e 2 têm uma calibração e um teste. O 1 tem mais ou menos o tamanho do X7a; o 2, o do agregado do X2, em
que 12 trimestres dependem da calibração de 2022. Neles, $X$ segue exatamente a beta-binomial, e o tamanho exato de cada
teste é a soma de $P(X = x)$ nos $x$ em que ele rejeita; o Monte Carlo serve de conferência. O cenário 3 imita H5 no
X7c, com quatro faixas, cada uma com a sua calibração, e compara a regra antiga, (i), com Holm sobre os p binomiais,
(ii), e com Holm sobre os p beta-binomiais, (iii), que é a regra atual.

In [3]:
cenarios = {nome: (n1, m, R_G1) for nome, (n1, m) in CENARIOS_G1.items()}
cenarios |= {f"3. faixa {j}": (n1, m, R_H5) for j, (n1, m) in enumerate(FAIXAS_H5, start=1)}
pd.DataFrame(
    {
        nome: {"n1, óbitos na parte B": num(n1), "m, óbitos no teste": num(m), "réplicas": num(r)}
        for nome, (n1, m, r) in cenarios.items()
    }
).T

,"n1, óbitos na parte B","m, óbitos no teste",réplicas
1. Tamanho do X7a,2.900,1.400,20.000
"2. Agregado do X2, uma calibração",2.640,16.440,20.000
3. faixa 1,200,100,200.000
3. faixa 2,500,250,200.000
3. faixa 3,1.100,550,200.000
3. faixa 4,900,450,200.000


In [4]:
rng = np.random.default_rng(SEMENTE)
linhas = []
for cenario, (n1, m) in CENARIOS_G1.items():
    x = sorteia_liberados(rng, n1, m, R_G1)
    testes = {"binomial": partial(p_binomial, m=m), "beta-binomial": partial(p_betabinomial, n1=n1, m=m)}
    for teste, p in testes.items():
        linhas.append((cenario, teste, tamanho_exato(n1, m, p), int(np.sum(p(x) < NIVEL)), R_G1))

p_bin, p_bb = np.empty((R_H5, len(FAIXAS_H5))), np.empty((R_H5, len(FAIXAS_H5)))
for j, (n1, m) in enumerate(FAIXAS_H5):
    x = sorteia_liberados(rng, n1, m, R_H5)
    p_bin[:, j], p_bb[:, j] = p_binomial(x, m), p_betabinomial(x, n1, m)
holm_binomial = holm_rejeita(p_bin).any(axis=1)
regras = {
    "(i) regra antiga: Holm ou alguma faixa com p < 0,05, binomial": holm_binomial | (p_bin < NIVEL).any(axis=1),
    "(ii) só Holm, binomial": holm_binomial,
    "(iii) Holm, beta-binomial (regra atual)": holm_rejeita(p_bb).any(axis=1),
}
cenario = "3. H5 com quatro faixas (como no X7c)"
linhas += [(cenario, regra, None, int(rejeitou.sum()), R_H5) for regra, rejeitou in regras.items()]

## Resultados

In [5]:
def taxa(rejeicoes, n):
    # porcentagem exata, arredondada com metade para cima
    valor = (Decimal(rejeicoes) * 100 / Decimal(n)).quantize(Decimal("0.1"), rounding=ROUND_HALF_UP)
    return f"{valor}%".replace(".", ",")


def erro_padrao(rejeicoes, n):
    r = rejeicoes / n
    return num(100 * math.sqrt(r * (1 - r) / n), 2) + " p.p."


pd.DataFrame(
    [
        {
            "cenário": cenario,
            "teste ou regra": teste,
            "tamanho exato": "-" if exato is None else pct(exato),
            "réplicas com rejeição": f"{num(rejeicoes)} de {num(n)}",
            "taxa de Monte Carlo": taxa(rejeicoes, n),
            "erro-padrão de Monte Carlo": erro_padrao(rejeicoes, n),
        }
        for cenario, teste, exato, rejeicoes, n in linhas
    ]
).set_index(["cenário", "teste ou regra"])

tamanho exato  \
cenário                               teste ou regra                                                                
1. Tamanho do X7a                     binomial                                                              7,58%   
                                      beta-binomial                                                         4,36%   
2. Agregado do X2, uma calibração     binomial                                                             26,11%   
                                      beta-binomial                                                         4,88%   
3. H5 com quatro faixas (como no X7c) (i) regra antiga: Holm ou alguma faixa com p < 0,05, binomial             -   
                                      (ii) só Holm, binomial                                                    -   
                                      (iii) Holm, beta-binomial (regra atual)                                   -   

                                                                                                    réplicas com rejeição  \
cenário                               teste ou regra                                                                        
1. Tamanho do X7a                     binomial                                                            1.433 de 20.000   
                                      beta-binomial                                                         797 de 20.000   
2. Agregado do X2, uma calibração     binomial                                                            5.219 de 20.000   
                                      beta-binomial                                                       1.011 de 20.000   
3. H5 com quatro faixas (como no X7c) (i) regra antiga: Holm ou alguma faixa com p < 0,05, binomial     54.560 de 200.000   
                                      (ii) só Holm, binomial                                            22.101 de 200.000   
                                      (iii) Holm, beta-binomial (regra atual)                            7.785 de 200.000   

                                                                                                    taxa de Monte Carlo  \
cenário                               teste ou regra                                                                      
1. Tamanho do X7a                     binomial                                                                     7,2%   
                                      beta-binomial                                                                4,0%   
2. Agregado do X2, uma calibração     binomial                                                                    26,1%   
                                      beta-binomial                                                                5,1%   
3. H5 com quatro faixas (como no X7c) (i) regra antiga: Holm ou alguma faixa com p < 0,05, binomial               27,3%   
                                      (ii) só Holm, binomial                                                      11,1%   
                                      (iii) Holm, beta-binomial (regra atual)                                      3,9%   

                                                                                                    erro-padrão de Monte Carlo  
cenário                               teste ou regra                                                                            
1. Tamanho do X7a                     binomial                                                                       0,18 p.p.  
                                      beta-binomial                                                                  0,14 p.p.  
2. Agregado do X2, uma calibração     binomial                                                                       0,31 p.p.  
                                      beta-binomial                                                                  0,15 p.p.  
3. H5 com quatro faixas (como no X7c) 

Sem deriva, o binomial rejeitaria 7,58% das vezes com o tamanho do X7a e 26,11% no agregado do X2, quando deveria ficar
em no máximo 5%. O beta-binomial fica em 4,36% e 4,88%. No cenário 1, as taxas de Monte Carlo ficaram abaixo dos
tamanhos exatos nos dois testes (7,2% e 4,0%), e as duas saem das mesmas 20.000 réplicas; o número que vale é o exato.

Em H5, a regra antiga daria uma refutação falsa em 27,3% das réplicas. Só com Holm, a taxa cai para 11,1%, porque o
binomial continua rejeitando demais em cada faixa, e com Holm sobre o beta-binomial, a regra atual, fica em 3,9%.